In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from datetime import datetime
from pyspark.sql import functions as F

spark=SparkSession.builder \
    .appName("Atica") \
    .enableHiveSupport() \
    .config(
      "spark.jars.packages",
      "org.apache.spark:spark-avro_2.12:3.3.2") \
    .getOrCreate()

data = [
    (1, "Hotel_A", "2024-01-01", "120.50", "2024-01-01 10:15:00"),
    (2, "Hotel_A", "2024-01-02", "150", "2024-01-02 11:45:30"),
    (3, "Hotel_B", "2024-01-01", "200.00", "2024-01-01 09:10:00"),
    (4, "Hotel_B", None, "180.75", "2024-01-03 18:30:00"),
    (5, "Hotel_C", "2024-01-03", None, "2024-01-03 20:00:00"),
    (6, "Hotel_C", "2024-01-04", "90", None)
]

schema = StructType([
    StructField("booking_id", IntegerType(), True),
    StructField("hotel", StringType(), True),
    StructField("booking_date", StringType(), True),
    StructField("amount", StringType(), True),
    StructField("created_ts", StringType(), True)
])

df = spark.createDataFrame(data, schema)
df.show(truncate=False)
df.printSchema()

In [ ]:
#Q1. Convert amount column from string to decimal(10,2).
#also handle null safely
from pyspark.sql.functions import *
from pyspark.sql.types import DecimalType

df_casted=df.withColumn('casted_amount',col('amount').cast(DecimalType(10,2)))
df_casted.printSchema()

In [ ]:
#Q2. Convert booking_date from string to date type.
#Difference between to_date() and cast(date)- cast is in general function which assumes data is already in dd-mm-yyyy format
#while to_date is used with timestamp or any string type of date which is not in dd-m-yyyy format. also it keeps null values as it is
date_casted=df.withColumn('casted_date',to_date('booking_date'))
#date_casted=df.withColumn('casted_date',col('booking_date').cast('date'))
date_casted.printSchema()

In [ ]:
#Q3. Convert created_ts to timestamp type.
#use to_timestamp -- by using this we will be able to keep null values as it is.
ts_casted=df.withColumn('casted_ts',to_timestamp('created_ts'))
ts_casted.printSchema()

In [ ]:
#Q4. Create a new column booking_year from booking_date.
#use date() --it also keeps null intact
year_from_date=df.withColumn('casted_date',to_date('booking_date'))\
                 .withColumn('booking_year',year('casted_date'))
year_from_date.show()

In [ ]:
#Q5. Create a column booking_month_name (January, February, etc.)
#use date_format - geenric function to format date in any pattern
month_from_date=df.withColumn('casted_date',to_date('booking_date'))\
                  .withColumn('booking_month',date_format(col('casted_date'),'MMMM')).show()

In [ ]:
#Q6. Replace NULL amount with 0.00 and NULL booking_date with '1970-01-01'.
#coalesce() or when() can be used here
fill_null_df=df.withColumn("amount",coalesce(col("amount").cast("decimal(10,2)"), F.lit(0.00)))
              .withColumn("booking_date",coalesce(to_date("booking_date"), F.lit("1970-01-01").cast("date"))).show()

In [ ]:
#Q7. Create a column is_weekend based on booking_date.
#if Saturday / Sunday → true , Else → false
day_from_date=df.withColumn('casted_date',to_date('booking_date'))\
                .withColumn('day_of_year',date_format(col('casted_date'),'dd'))\
                .withColumn('isweekend',when(dayofweek('day_of_year')=='06','True')
                                       .when(dayofweek('day_of_year')=='07','True').otherwise('False')).show()
#.withColumn("is_weekend",F.dayofweek("casted_date").isin([1, 7]))

In [ ]:
#Q8. Find number of bookings per hotel per day.
#expect: hotel | booking_date | booking_count (group by and count)
bookings_per_day=df.groupBy('hotel','booking_date').count().alias('booking_count').show()

#more correct way could have been
'''
df.groupBy("hotel", "booking_date") \
  .agg(F.count("*").alias("booking_count"))
'''

In [ ]:
#Q9. Create a column booking_delay_days
#Difference between booking_date and created_ts date.
#use date_diff()
date_diff=df.withColumn('created_ts',to_timestamp('created_ts'))\
            .withColumn('casted_ts',to_date('created_ts'))\
            .withColumn('booking_date',to_date('booking_date'))\
            .withColumn('date_diff',datediff(col('casted_ts'),col('booking_date'))).show()

In [ ]:
#Q10. Filter records where:
'''
booking_date is not NULL
amount > 100
booking year = 2024'''

filtered_df=df.withColumn("booking_date", to_date("booking_date")) \
              .withColumn("amount", col("amount").cast("decimal(10,2)")) \
              .filter(
                  (col("booking_date").isNotNull()) &
                  (col("amount") > 100) &
                  (year("booking_date") == 2024)
              ).show()

## Set-2: (Join,Windows,Partitioning,Deduplication)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.types import DecimalType

bookings_data = [
    (1, "H001", "2024-01-01", 120.5),
    (2, "H001", "2024-01-02", 150.0),
    (3, "H002", "2024-01-01", 200.0),
    (4, "H002", "2024-01-02", 180.0),
    (5, "H003", "2024-01-01", 90.0),
    (6, "H003", "2024-01-01", 95.0),   # duplicate day booking
]

bookings_schema = ["booking_id", "hotel_id", "booking_date", "amount"]

bookings_df = spark.createDataFrame(bookings_data, bookings_schema)

hotels_data = [
    ("H001", "Hotel_A", "Mumbai"),
    ("H002", "Hotel_B", "Delhi"),
    ("H003", "Hotel_C", "Pune")
]

hotels_schema = ["hotel_id", "hotel_name", "city"]

hotels_df = spark.createDataFrame(hotels_data, hotels_schema)

In [ ]:
bookings_df.show()
hotels_df.show()

In [ ]:
#Q1. Perform an inner join between bookings and hotels.
join_df=bookings_df.alias('b').join(hotels_df.alias('h'),on='hotel_id',how='inner')\
        .select(col('b.booking_id'),col('h.hotel_name'),col('h.city'),col('b.booking_date'),col('b.amount'))\
        .show()

In [ ]:
#Q2. Convert the join to a left join and explain:
#“I use a left join when I want to retain all fact records even if dimension data is missing, 
#especially in cases of late-arriving dimensions or incomplete reference data.”
left_join_df=bookings_df.alias('b').join(hotels_df.alias('h'),on='hotel_id',how='left')\
        .select(col('b.booking_id'),col('h.hotel_name'),col('h.city'),col('b.booking_date'),col('b.amount'))\
        .show()

In [ ]:
#Q3. Broadcast the hotels table explicitly and explain why this improves performance.
#primarily to optimize join operations when one dataset is significantly smaller than the other. 
#This prevents data shuffling across the network and improves performance. 
from pyspark.sql.functions import broadcast
broadcast_df=bookings_df.join(broadcast(hotels_df),on='hotel_id',how='inner')
broadcast_df.explain("formatted")
broadcast_df.show()

#“I would not broadcast if the dimension table is large or grows unpredictably, as it can cause executor OOM errors.”

In [ ]:
#Q4. Find daily total revenue per hotel.
#hotel_id | booking_date | daily_revenue
#“I aggregate before joins wherever possible to reduce data volume and shuffle.”
daily_revenue_df = (
    bookings_df
    .groupBy("hotel_id", "booking_date")
    .agg(sum("amount").alias("daily_revenue"))
)

daily_revenue_df.show()

In [ ]:
#Q5. Identify duplicate bookings per hotel per day.
#window functions + count
from pyspark.sql.window import Window
window_spec=Window.partitionBy('hotel_id','booking_date')
duplicates=bookings_df.withColumn('booking_count',count('*').over(window_spec))\
                      .filter(col('booking_count')>1)
                      .show()

In [ ]:
#Q6. Keep only latest booking per hotel per day based on booking_id.
window_spec = (
    Window
    .partitionBy("hotel_id", "booking_date")
    .orderBy(desc("booking_id"))
)

latest_df = (
    bookings_df
    .withColumn("rn", row_number().over(window_spec))
    .filter(col("rn") == 1)
    .drop("rn")
)

latest_df.show()

In [ ]:
#Q7. Add a column prev_day_revenue using window functions.
daily_rev = (
    bookings_df
    .groupBy("hotel_id", "booking_date")
    .agg(sum("amount").alias("daily_revenue"))
)

window_spec = Window.partitionBy("hotel_id").orderBy("booking_date")

final_df = daily_rev.withColumn(
    "prev_day_revenue",
    F.lag("daily_revenue").over(window_spec)
)

final_df.show()

In [ ]:
#Q8. Explain and demonstrate the difference between:
#repartition(5)
#coalesce(5)
#“Repartition causes a full shuffle and should be avoided in tight pipelines.
#Coalesce is preferred when reducing partitions after heavy transformations.”
df = spark.range(0, 1000, step=1, numPartitions=20)
print(f"Initial number of partitions: {df.rdd.getNumPartitions()}")

#repartition
df_repartitioned=df.repartition(5)
print(f"partitions after repartition(5): {df_repartitioned.rdd.getNumPartitions()}")
df_repartitioned.write.mode("overwrite").format("noop").save()

#coalesce
df_coalesced = df.coalesce(5)

print(f"Partitions after coalesce(5): {df_coalesced.rdd.getNumPartitions()}")

In [ ]:
#Q9. Count number of partitions before and after repartition.
print(f"Partitions after coalesce(5): {df_coalesced.rdd.getNumPartitions()}")
print(f"partitions after repartition(5): {df_repartitioned.rdd.getNumPartitions()}")

In [ ]:
#Q10. Scenario Question (Must Answer Structurally)
#“Bookings table is 500 GB. Join with a 5 MB hotels table is slow. What will you do?”
'''
Broadcast the 5 MB hotels table
Ensure bookings table is partitioned (e.g., by date)
Push filters early for partition pruning
Validate shuffle reduction in Spark UI
Tune executor memory and cores on Dataproc
Aggregate before join if possible

⭐ Closing Line

“I optimize data movement first, infra second.”
'''

## Set-3: (Aggregations,Skew,Shuffle, Performance thinking)

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *

data = [
    ("H001", "2024-01-01", "Mumbai", 120.0),
    ("H001", "2024-01-01", "Mumbai", 150.0),
    ("H001", "2024-01-02", "Mumbai", 200.0),
    ("H002", "2024-01-01", "Delhi", 300.0),
    ("H002", "2024-01-01", "Delhi", 180.0),
    ("H003", "2024-01-01", "Pune", 90.0),
    ("H003", "2024-01-02", "Pune", 95.0),
    ("H003", "2024-01-02", "Pune", 100.0)
]

schema = ["hotel_id", "booking_date", "city", "amount"]

df = spark.createDataFrame(data, schema)
df.show()

In [ ]:
#Q1. Calculate daily revenue per hotel per city.
#o./p:hotel_id | city | booking_date | total_revenue
daily_rev_per_city=df.groupBy('hotel_id','booking_date','city').agg(
                     sum('amount').alias('total_revenue')
                    ).select(col('hotel_id'),col('city'),col('booking_date'),col('total_revenue')).show()

In [ ]:
#Q2. Find top 2 revenue-generating hotels per day.
daily_rev = (
    df.groupBy("hotel_id", "city", "booking_date")
      .agg(sum("amount").alias("daily_revenue"))
)

window_spec = Window.partitionBy("booking_date").orderBy(col("daily_revenue").desc())

top_2 = (
    daily_rev.withColumn("rank", dense_rank().over(window_spec))
             .filter(col("rank") <= 2)
)

top_2.show()
#“First I aggregate, then I apply ranking. Ranking on raw rows gives incorrect business results.”

In [ ]:
#Q3. Compute running cumulative revenue per hotel ordered by booking_date.
window_spec = (
    Window.partitionBy("hotel_id")
          .orderBy("booking_date")
          .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

running_cum_sum = df.withColumn(
    "cumulative_revenue",
    sum("amount").over(window_spec)
)

running_cum_sum.show()
#“Cumulative metrics must always partition by business key and order by time.”

In [ ]:
#Q4. Identify hotels contributing more than 40% of total daily revenue.
daily_hotel_rev = (
    df.groupBy("hotel_id", "booking_date")
      .agg(sum("amount").alias("daily_hotel_revenue"))
)

daily_total_rev = (
    daily_hotel_rev.withColumn(
        "total_daily_revenue",
        sum("daily_hotel_revenue").over(Window.partitionBy("booking_date"))
    )
)

hotels_40 = (
    daily_total_rev.withColumn(
        "contri_percentage",
        col("daily_hotel_revenue") / col("total_daily_revenue")
    )
    .filter(col("contri_percentage") > 0.4)
)

hotels_40.show()


In [ ]:
#Q5. Cache the dataset optimally and explain:
df.cache()
df.count()
'''we should use cache when we have a very small input,which fits comfortably in executors memory, with default storage levels as MEMORY_ONLY,
so instead of persisting it to the disk, we tend to do caching, to repeatedly use in multiple stages and jobs
“I cache only when a DataFrame is reused multiple times and fits in executor memory.
Otherwise, caching can increase GC pressure and slow jobs.”'''

In [ ]:
#Q6. Detect data skew in this dataset.
skew_df=df.groupBy('hotel_id').agg(
          count('*').alias('count_of_hotels')).show()
'''
“Salting converts one large skewed shuffle into multiple smaller balanced shuffles.”
data can be skewed if any of the hotel is higher than avergae count of all hotels.
and if data is 80% at hotel-01, then it is severely skewed, which may cause worker node slowness due to data is mostly partitioned
on 1 worker only.
then OOM erros, to mitigate use salting,skewed joins optimizations and filtering techniques.
'''

In [ ]:
#Q7. Fix skew using salting technique.
'''Expectation
Add salt column
Repartition
Re-aggregate'''
salted_df = df.withColumn("salt", (rand() * 10).cast("int"))

salted_agg = (
    salted_df.groupBy("hotel_id", "salt")
             .agg(sum("amount").alias("partial_sum"))
)

final_agg = (
    salted_agg.groupBy("hotel_id")
              .agg(sum("partial_sum").alias("total_revenue"))
)

final_agg.show()


In [ ]:
#Q8. Optimize shuffle using Spark configs (Dataproc Context)
spark.conf.set("spark.sql.shuffle.partitions", "8")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
'''
Explanation (Say This)

“Shuffle partitions should be proportional to total executor cores. AQE dynamically reduces partitions and handles skew at runtime, which is extremely useful on Dataproc where data volume varies.”

Why this matters on Dataproc

Fewer partitions → less overhead

AQE avoids over-parallelization

Saves cost + execution time
'''

In [ ]:
#Q9. Why wide transformations are expensive (With Example)
df.groupBy("hotel_id").sum("amount")
'''
What happens internally
Data shuffled across network
Written to disk
Pulled by reducers

Causes:
Network I/O
Disk spill
Executor imbalance

Interview-grade explanation
“Wide transformations trigger shuffle, which is the most expensive operation in Spark due to network and disk I/O. 
My goal is always to reduce shuffle size before scaling infrastructure.”
'''

In [ ]:
#Q10. 500 GB Job Fails – How Do You Fix It? (MOST IMPORTANT)
'''Step 1: Spark UI
Identify long stages
Check skewed tasks

Step 2: Data-level Fix
Reduce columns early
Filter early
Fix skew (salting / broadcast)

Step 3: Spark-level Fix
Increase shuffle partitions correctly
Enable AQE
Use broadcast joins

Step 4: Infra-level Fix (Dataproc)
Increase executors, not executor size
Enable autoscaling
Use preemptible workers

Closing Line (Say This)
“I always fix data shape and shuffle first. Scaling infra without fixing shuffle only increases cost.”'''

## Set 4 :(File Formats, Partitioning, Incremental Loads, BigQuery Integration)

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *

data = [
    (1, "H001", "2024-01-01", "Mumbai", 120.0),
    (2, "H001", "2024-01-02", "Mumbai", 150.0),
    (3, "H002", "2024-01-01", "Delhi", 200.0),
    (4, "H002", "2024-01-02", "Delhi", 180.0),
    (5, "H003", "2024-01-01", "Pune", 90.0),
    (6, "H003", "2024-01-03", "Pune", 110.0)
]

schema = ["booking_id", "hotel_id", "booking_date", "city", "amount"]

df = spark.createDataFrame(data, schema)
df = df.withColumn("booking_date", F.to_date("booking_date"))
df.show()

In [ ]:
#Q1. Write this DataFrame to GCS in Parquet format, partitioned by booking_date.
gcs_output_path="gs://dataproc-temp-us-west1-19745307921-5piujqir/pyspark-tutorials/"
df.write\
  .mode('overwrite')\
  .partitionBy('booking_date')\
  .parquet(gcs_output_path)
print(f"Data successfully written to {gcs_output_path}, partitioned by booking_date.")

#before writing to production use this --> to avoid small files in gcs location
#“Partitioning by booking_date enables partition pruning, which reduces scan cost and improves query performance downstream,
#especially for BigQuery external reads or Spark reads.”
df.repartition("booking_date")

In [ ]:
pq_df=spark.read.parquet("gs://dataproc-temp-us-west1-19745307921-5piujqir/pyspark-tutorials/booking_date=2024-01-01/part-00000-67378920-e500-4832-8c95-b68ec932ae7a.c000.snappy.parquet")
pq_df.printSchema()
pq_df.show()

In [ ]:
#Q2. Rewrite Q1 to store data in Avro format.
gcs_output_path="gs://dataproc-temp-us-west1-19745307921-5piujqir/avro_files/"
df.write \
  .format("avro") \
  .mode("overwrite") \
  .partitionBy("booking_date") \
  .save(gcs_output_path)
#Avro is preferred in use cases that require robust schema evolution,
#efficient row-based storage, and streaming friendliness, such as real-time data ingestion pipelines and messaging systems. 
'''
“Avro is row-based and supports strong schema evolution, making it suitable for streaming ingestion, 
CDC pipelines, and inter-service data exchange. 
Parquet is columnar and preferred for analytics.”
'''


In [ ]:
#Q3. Read back only data for booking_date = '2024-01-02'.
df_filtered = spark.read.parquet(
    "gs://dataproc-temp-us-west1-19745307921-5piujqir/pyspark-tutorials/"
).filter(F.col("booking_date") == "2024-01-02")
df_filtered.show()

In [ ]:
#Q4. Append new booking data without reprocessing historical data.
from delta.tables import DeltaTable
'''
Scenario
New data arrives daily in GCS

Expectation
Append mode
Idempotent design
'''
#assuming new data arriving at below location
daily_new_data='gs://pyspark-tutorials/new/'
#historical data lying here at below location
historical_data='gs://dataproc-temp-us-west1-19745307921-5piujqir/input_files/'
#now just read the new data
df=spark.read.parquet(daily_new_data)
df.write\
  .mode('append')\
  .parquet(historical_data)

'''#we can use delta table also which supports merge (upserts)
# Perform a MERGE (upsert) operation
delta_table_path = "gs://your-bucket/processed/bookings_delta/"

# Read daily data
daily_df = spark.read.parquet(daily_data_path)
deltaTable = DeltaTable.forPath(spark, delta_table_path)

deltaTable.merge(
    daily_df.alias("source"),
    "target.booking_id = source.booking_id" # Match condition
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()'''

In [ ]:
#Q5. Design an incremental load using booking_date as watermark.
#Q6. Deduplicate data before writing using booking_id.
'''Explain
How to track last processed date
Where to store state (BQ / GCS / metadata table)
This is a common interview question.'''
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from datetime import datetime, timedelta

def run_incremental_load(spark, last_processed_date):
    # Define the high watermark for this current run (e.g., now or a fixed time)
    # This example uses a fixed 'now' for demonstration
    current_high_watermark = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    
    # 1. Read the source data
    source_df = spark.read.parquet("path/to/source/data")

    # 2. Apply the incremental filter using the 'last_processed_date'
    last_processed_date = "2024-01-02"
    if last_processed_date:
        print(f"Filtering data booked after: {last_processed_date}")
        source_df = source_df.filter(F.col("booking_date") > F.lit(last_processed_date))
    else:
        print("No previous state found, loading all data.")

    # 3. Perform necessary transformations (e.g., deduplication, cleaning)
    transformed_df = source_df.dropDuplicates(["booking_id"])

    # 4. Write to the destination (e.g., BigQuery, Data Lake)
    # Consider using 'append' or 'merge' (if using Delta Lake or similar lakehouse formats)
    transformed_df.write \
        .mode("append") \
        .parquet("path/to/destination/datalake")
    #
    new_max_date = incremental_df.agg(
    F.max("booking_date")
    ).collect()[0][0]

    # 5. Return the new high watermark to be persisted
    return current_high_watermark

#then for second option , I will use dedicated metadata table in bq
'''
How it works:
You maintain a simple table with columns like job_name, last_run_timestamp, and status. 
Your PySpark job starts by reading this table to get the previous watermark and updates it upon successful completion. 

“In batch pipelines, watermarks are implemented logically, not via Spark streaming.
I track the maximum processed date in a metadata store and filter new records in every run.
This ensures idempotency and avoids reprocessing historical data.”
'''

In [ ]:
#Q7. Write aggregated daily revenue per hotel into BigQuery.
df.show()
project_id='unified-canyon-477712'
dataset_id='bq_stage'
table_id='hotel_data'

temporary_gcs_bucket = "gs://dataproc-temp-us-west1-19745307921-5piujqir/"
spark.conf.set('temporaryGcsBucket', temporary_gcs_bucket)

agg_df=df.groupBy('hotel_id','booking_date').agg(
         sum('amount').alias('daily_revenue')
        )
agg_df.write\
      .format('bigquery')\
      .option("table", f"{project_id}.{dataset_id}.{table_id}")\
      .mode('overwrite')\
      .save()
print(f"Successfully wrote DataFrame to BigQuery table: {dataset_id}.{table_id}")

In [ ]:
#Q8. Explain append vs overwrite vs truncate in BigQuery writes.
'''
✅ Interview-Grade Answer

Append: Adds new rows; safest for incremental loads

Overwrite: Replaces table or partition; risky without filters

Truncate: Empties table; rarely used in production

Add:

“In production, I prefer partition-level overwrite over full table overwrite.”
'''

In [ ]:
#Q9. Optimize BigQuery table written from Spark

'''
to optimize data written in bq,I will tryy to implement
partitioning , clustering on column level
then as spark checkpoints intermediate data into gcs like storage, I will try to reduce the file_size of data,
so it will not cause overhead
“Partitioning reduces scan cost, 
clustering improves predicate filtering, and avoiding small files improves BigQuery load efficiency.”
'''
df.write \
  .format("bigquery") \
  .option("table", "project.dataset.table") \
  .option("partitionField", "event_date") \
  .option("clusteredColumns", "user_id,session_id") \
  .mode("append") \
  .save()

In [ ]:
#Q10. Real Interview Scenario (Very Important)
#“Dataproc job reruns accidentally and duplicates data in BigQuery. How do you prevent this?”
'''“I design pipelines to be idempotent using business keys and partition-aware writes. 
For BigQuery, I either use MERGE statements or partition overwrite. 
Airflow retries are safe because reruns do not create duplicates.”'''

## Set 5 : (End-to-End Pipelines, Dataproc Ops, Airflow, Cost & Failure Handling)

🔸 Business Context (Very Important)

You work for a hospitality analytics platform.

Every day:

Booking data lands in GCS

Size: 200–500 GB/day

You must:

Transform it using PySpark on Dataproc

Aggregate revenue

Load results into BigQuery

Pipeline is triggered by Airflow

In [ ]:
#Q1. Design an end-to-end Dataproc PySpark pipeline.

'''Expected Answer Structure (Say it like this):

Ingestion
Raw booking files land in GCS (Parquet/CSV)
Folder structure:
    gs://bucket/raw/bookings/booking_date=YYYY-MM-DD/


Processing
Airflow triggers Dataproc job
PySpark reads only required partitions
Transformations + aggregations

Storage
Write curated data back to GCS (Parquet)
Load aggregated results to BigQuery

Orchestration
Airflow DAG controls retries, alerts, dependencies

Interview Signal:
Clear separation of raw → processed → analytics layers.'''

In [ ]:
#Q2. How would you trigger this pipeline using Airflow?

'''Expected Tools
DataprocCreateClusterOperator (or ephemeral cluster)
DataprocSubmitJobOperator
DataprocDeleteClusterOperator

Best Practice Answer
Use ephemeral clusters
Create → submit → delete in same DAG

Why interviewer likes this
Cost control
Clean infra lifecycle'''

In [ ]:
#Q3. What happens if a Dataproc worker node fails mid-job?

'''Correct Explanation
Spark tasks are retried automatically
Executors on failed node are lost
Tasks are rescheduled on other executors
Job continues unless driver fails

Follow-up
What if master fails?

Answer
Job fails
Needs restart
Reason why driver stability matters'''

In [ ]:
#Q4. How do you handle job retries without duplicating data?

'''This is a MUST-KNOW question.

Your answer must include:
Idempotent design
Deduplication keys
Partition overwrite
BigQuery MERGE

Strong Line
“Retries should re-run safely without side effects.”'''

In [ ]:
#Q5. Dataproc job is slow only on certain days. Why?

'''Expected Diagnosis Thinking

Possible reasons:
Data skew (holiday bookings spike)
Large shuffles
Bad partitioning
Executor memory pressure
Too few shuffle partitions

Action Plan
Check Spark UI
Identify skewed keys
Enable AQE
Tune partitions
Add autoscaling'''

In [ ]:
#Q6. How do you scale Dataproc clusters dynamically?
'''
Expected Concepts
Autoscaling policy
Scale based on YARN pending memory
Min/max workers

Key Interview Line
“We scale based on workload, not fixed infra.”'''

In [ ]:
#Q7. How do you reduce Dataproc costs?

'''You must mention at least 5
Ephemeral clusters
Preemptible workers
Right-sized executors
Avoid over-partitioning
Partition pruning
Avoid caching unnecessarily

Efficient file formats (Parquet)
If you say all → strong signal'''

In [ ]:
#Q8. Explain a production failure scenario and how you handled it.

'''Sample Answer (Use This Pattern)

“One of our revenue aggregation jobs failed intermittently.
After checking Spark UI, we found heavy data skew on a single hotel_id.
We fixed it by salting the key and enabling AQE.
Post-fix, job runtime reduced by 60%.”

Interviewers LOVE this structure:
Problem → Diagnosis → Fix → Result'''

In [ ]:
#Q9. What metrics do you monitor for Dataproc jobs?

'''Expected Monitoring Areas
Job duration
Executor CPU & memory
Shuffle read/write size
Failed tasks
Cost per run
Airflow SLA misses

Bonus
Stackdriver / Cloud Monitoring'''

In [ ]:
#Q10. FINAL INTERVIEW QUESTION (MOST IMPORTANT)

'''“Why Dataproc + PySpark for this use case instead of BigQuery SQL only?”

Perfect Answer
“BigQuery is excellent for analytics, but Dataproc gives us flexibility to handle complex transformations, custom logic, incremental processing, and cost-efficient batch workloads.
We use PySpark for heavy processing and BigQuery for serving analytics.”

This shows architectural maturity.'''

In [1]:
spark.stop()